# 🦥 Unsloth AI Puzzles — Google TPU (JAX & XLA)
Google TPU architecture leveraging JAX NamedSharding, XLA compilation, and vectorization.


---
## Hardware Check
Verify the Google TPU connection and hardware specifications.


In [1]:
try:
    import jax.tools.colab_tpu
    jax.tools.colab_tpu.setup_tpu()
except Exception:
    pass

import jax
import os
os.environ.pop("XLA_FLAGS", None)
os.environ.pop("JAX_PLATFORMS", None)

print(f"JAX Backend: {jax.default_backend().upper()}")
devices = jax.devices()
print(f"Detected {len(devices)} device(s):")
for d in devices:
    print(f"  - {d.device_kind} (ID: {d.id})")


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


JAX Backend: TPU
Detected 1 device(s):
  - TPU v5 lite (ID: 0)


---
## Task A (NF4 Dequantization)
Vectorized jax.numpy processing.


In [2]:
import jax
import jax.numpy as jnp
from functools import partial

# Standard NF4 exact values
NF4_VALUES = jnp.array([
    -1.0, -0.6961928, -0.52507305, -0.39491749, -0.28444138, -0.18477343, -0.09105004, 0.0,
    0.0795803, 0.1609302, 0.2461123, 0.33791524, 0.44070983, 0.562617, 0.72295684, 1.0
], dtype=jnp.bfloat16)

@partial(jax.jit, static_argnames=("block_size",))
def dequantize_nf4_jax(weight_uint8, absmax, block_size=64):
    """
    JAX / XLA native NF4 dequantization.
    This replaces the Triton kernel from the official challenge.
    """
    # 1. Unpack 8-bit into two 4-bit values (nibbles)
    # Bits 0-3: first value, Bits 4-7: second value
    low_nibble = jnp.bitwise_and(weight_uint8, jnp.uint8(0x0F))
    high_nibble = jnp.right_shift(weight_uint8, jnp.uint8(4))
    
    # 2. Interleave to form (N*2,) int32 array for lookup
    # e.g. [low0, high0, low1, high1, ...]
    unpacked = jnp.stack([low_nibble, high_nibble], axis=1).flatten()
    
    # 3. Table lookup against exact NF4 floats
    dequantized = jnp.take(NF4_VALUES, unpacked)
    
    # 4. Scale by block-wise absmax
    dequantized_blocks = dequantized.reshape(-1, block_size)
    scaled = dequantized_blocks * absmax[:, None]
    
    return scaled.flatten()

if __name__ == "__main__":
    print("Testing JAX NF4 Dequantization (Task A)...")
    N_bytes = 4096
    block_size = 64
    
    key = jax.random.PRNGKey(0)
    key1, key2 = jax.random.split(key)
    
    # Mock data
    weight_uint8 = jax.random.randint(key1, (N_bytes,), 0, 256, dtype=jnp.uint8)
    absmax = jax.random.uniform(key2, (N_bytes * 2 // block_size,), dtype=jnp.bfloat16)
    
    # Compile and run
    out = dequantize_nf4_jax(weight_uint8, absmax)
    print("Dequantized shape:", out.shape)
    print("Dequantized preview:", out[:8])
    print(" JAX NF4 kernel compiled and executed successfully")


Testing JAX NF4 Dequantization (Task A)...
Dequantized shape: (8192,)
Dequantized preview: [0.390625 0.0961914 0.0629883 0.0311279 0.390625 0.0629883 0.0629883 0]
 JAX NF4 kernel compiled and executed successfully


---
## Task B (NamedSharding over TPU Topology)
True multi-core TPU sharding via jax.sharding.Mesh.


In [3]:
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from jax.experimental import mesh_utils
import time

def simulate_fsdp2_in_jax():
    print(" Initializing JAX Multi-Device Cluster...")
    devices = jax.devices()
    print(f"Detected Devices: {devices}")
    assert len(devices) >= 1, "Failed to detect JAX devices"

    # 1. Create a Hardware Mesh (equivalent to torch.distributed.init_process_group)
    # We lay out our available devices in a 1D grid named 'fsdp_axis'
    mesh = Mesh(mesh_utils.create_device_mesh((len(devices),)), axis_names=('fsdp_axis',))

    # 2. Define our Sharding Strategy (equivalent to FullyShardedDataParallel)
    # We tell JAX to shard the first dimension of our weight matrix across the 'fsdp_axis'
    fsdp_sharding = NamedSharding(mesh, P('fsdp_axis', None))
    
    # Replicated strategy (for inputs/outputs that every device needs a full copy of)
    replicated = NamedSharding(mesh, P(None, None))

    print("\n Generating massive weight matrix (Simulating 8B parameters)...")
    # We generate a large matrix on the TPU...
    key = jax.random.PRNGKey(0)
    W_full = jax.random.normal(key, (8192, 8192))
    
    # 3. Apply FSDP Sharding
    # This physically chops the matrix in half and sends 4096 rows to Device 0 and 4096 to Device 1
    W_sharded = jax.device_put(W_full, fsdp_sharding)
    
    print("Weight Matrix Sharding Profile:")
    jax.debug.visualize_array_sharding(W_sharded)
    print(f"Is it sharded? {W_sharded.is_fully_replicated == False}")

    # 4. Define our Training Step
    # In JAX, we just write normal math. XLA automatically inserts the NCCL All-Gather and Reduce-Scatter networking
    @jax.jit
    def fsdp_forward_backward(W, X):
        # Forward Pass (XLA automatically inserts All-Gather if needed)
        logits = jnp.dot(X, W.T)
        loss = jnp.sum(logits ** 2)
        
        # Backward Pass (XLA automatically calculates gradients and Reduce-Scatters them back to the mesh)
        dW = jax.grad(lambda w: jnp.sum(jnp.dot(X, w.T) ** 2))(W)
        return loss, dW

    # Dummy Input (replicated across all devices)
    X = jax.device_put(jax.random.normal(key, (128, 8192)), replicated)

    print("\n Running Distributed FSDP Forward/Backward pass...")
    start = time.perf_counter()
    loss, dW = fsdp_forward_backward(W_sharded, X)
    
    # Block until both devices finish
    loss.block_until_ready()
    dW.block_until_ready()
    end = time.perf_counter()
    
    print(f"Loss: {loss}")
    print("Gradient Matrix Sharding Profile (Notice how gradients are automatically sharded to match the weights):")
    jax.debug.visualize_array_sharding(dW)
    print(f"Completed distributed pass in {end - start:.4f}s")
    print(" Simulated FSDP2 JAX training successfully on Google TPU")

if __name__ == "__main__":
    simulate_fsdp2_in_jax()


 Initializing JAX Multi-Device Cluster...
Detected Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

 Generating massive weight matrix (Simulating 8B parameters)...
Weight Matrix Sharding Profile:


                         
                         
                         
                         
                         
          TPU 0          
                         
                         
                         
                         
                         

Is it sharded? False

 Running Distributed FSDP Forward/Backward pass...
Loss: 17124492288.0
Gradient Matrix Sharding Profile (Notice how gradients are automatically sharded to match the weights):


                         
                         
                         
                         
                         
          TPU 0          
                         
                         
                         
                         
                         

Completed distributed pass in 0.2226s
 Simulated FSDP2 JAX training successfully on Google TPU


---
## Task C (jax.jit compilation)
No-break XLA graph execution.


In [4]:
import jax
import jax.numpy as jnp
from functools import partial
import time

@partial(jax.jit, static_argnames=("is_causal",))
def flex_attention_jax(q, k, v, is_causal=True):
    """
    JAX Native Compiled Attention.
    Replaces torch.compile + flex_attention.
    XLA automatically fuses this entire function into a single TPU operator.
    """
    B, H, L, D = q.shape
    
    # 1. Q @ K.T
    scores = jnp.einsum('bhld,bhmd->bhlm', q, k)
    
    # 2. Scale
    scores = scores * (1.0 / jnp.sqrt(D))
    
    # 3. Causal Mask
    if is_causal:
        # Create lower triangular mask
        mask = jnp.tril(jnp.ones((L, L), dtype=bool))
        # Apply -inf where mask is False
        scores = jnp.where(mask, scores, -jnp.inf)
        
    # 4. Softmax (XLA fuses the subtraction of max for stability)
    max_scores = jnp.max(scores, axis=-1, keepdims=True)
    exp_scores = jnp.exp(scores - max_scores)
    probs = exp_scores / jnp.sum(exp_scores, axis=-1, keepdims=True)
    
    # 5. @ V
    out = jnp.einsum('bhlm,bhmd->bhld', probs, v)
    return out

if __name__ == "__main__":
    print("Testing JAX Fused Attention (Task C)...")
    B, H, L, D = 2, 4, 1024, 64
    
    key = jax.random.PRNGKey(0)
    q = jax.random.normal(key, (B, H, L, D), dtype=jnp.bfloat16)
    k = jax.random.normal(key, (B, H, L, D), dtype=jnp.bfloat16)
    v = jax.random.normal(key, (B, H, L, D), dtype=jnp.bfloat16)
    
    # 1. Trigger JIT Compilation (Warmup)
    print("Compiling JAX graph into XLA...")
    _ = flex_attention_jax(q, k, v).block_until_ready()
    
    # 2. Benchmark execution time
    print("Running benchmarks...")
    start = time.perf_counter()
    num_passes = 10
    for _ in range(num_passes):
        out = flex_attention_jax(q, k, v)
        out.block_until_ready()
    end = time.perf_counter()
    
    print(f" JAX Compiled Attention: {num_passes} passes in {end - start:.4f}s")
    print("Output shape:", out.shape)


Testing JAX Fused Attention (Task C)...
Compiling JAX graph into XLA...
Running benchmarks...
 JAX Compiled Attention: 10 passes in 0.0018s
Output shape: (2, 4, 1024, 64)


---
## Task E (Memory-Efficient Backprop)
Custom backward evaluation via jax.custom_vjp.


In [ ]:
import jax
import jax.numpy as jnp
import time

import functools

# -----------------------------------------------------------------------------
# 1. Functional Definition (Forward Pass)
# -----------------------------------------------------------------------------
@functools.partial(jax.custom_vjp, nondiff_argnums=(3,))
def chunked_cross_entropy(X, W, Y, chunk_size):
    """
    Computes cross entropy loss by chunking the sequence dimension
    to prevent materializing the full N x V logits matrix in memory.
    """
    N = X.shape[0]
    
    def scan_fn(carry, idx):
        # Slice chunks — handle partial last chunk
        actual_size = jnp.minimum(chunk_size, N - idx)
        X_chunk = jax.lax.dynamic_slice_in_dim(X, idx, chunk_size, axis=0)[:actual_size]
        Y_chunk = jax.lax.dynamic_slice_in_dim(Y, idx, chunk_size, axis=0)[:actual_size]
        
        # Forward pass for chunk
        logits = jnp.dot(X_chunk, W.T) # (C, V)
        
        # LogSumExp
        lse = jax.scipy.special.logsumexp(logits, axis=1) # (C,)
        
        # Target logits
        correct_logits = logits[jnp.arange(actual_size), Y_chunk]
        
        loss_chunk = lse - correct_logits
        return carry + jnp.sum(loss_chunk), None

    total_loss, _ = jax.lax.scan(scan_fn, 0.0, jnp.arange(0, N, chunk_size))
    return total_loss / N

# -----------------------------------------------------------------------------
# 2. VJP Rules (Vector-Jacobian Product) for Backward Pass
# -----------------------------------------------------------------------------
def ce_fwd(X, W, Y, chunk_size):
    # The forward pass returns the primary output AND the residuals needed for backward
    loss = chunked_cross_entropy(X, W, Y, chunk_size)
    return loss, (X, W, Y)

def ce_bwd(chunk_size, res, g):
    # Unpack residuals
    X, W, Y = res
    N, H = X.shape
    V, _ = W.shape
    
    def scan_fn_bwd(carry, idx):
        dW_acc = carry
        
        actual_size = jnp.minimum(chunk_size, N - idx)
        X_chunk = jax.lax.dynamic_slice_in_dim(X, idx, chunk_size, axis=0)[:actual_size]
        Y_chunk = jax.lax.dynamic_slice_in_dim(Y, idx, chunk_size, axis=0)[:actual_size]
        
        logits = jnp.dot(X_chunk, W.T)
        probs = jax.nn.softmax(logits, axis=-1)
        
        # Derivative of Cross Entropy: (probs - 1.0 for target class)
        d_logits = probs.at[jnp.arange(actual_size), Y_chunk].add(-1.0)
        
        # Scale by upstream gradient `g` and batch size `N`
        d_logits = d_logits * (g / N)
        
        # Matrix multiply backward pass
        dX_chunk = jnp.dot(d_logits, W) # (C, H)
        dW_chunk = jnp.dot(d_logits.T, X_chunk) # (V, H)
        
        return dW_acc + dW_chunk, dX_chunk

    # Scan sequentially accumulates dW, and stacks dX
    dW_total, dX_chunks = jax.lax.scan(scan_fn_bwd, jnp.zeros_like(W), jnp.arange(0, N, chunk_size))
    
    # Flatten dX chunks back to (N, H)
    dX_full = dX_chunks.reshape(-1, H)[:N]  # handle partial last chunk
    
    # Return gradients for (X, W, Y) - None for non-differentiable params
    return (dX_full, dW_total, None)

# Register the VJP functions
chunked_cross_entropy.defvjp(ce_fwd, ce_bwd)

# -----------------------------------------------------------------------------
# 3. Naive Implementation (For Verification)
# -----------------------------------------------------------------------------
def naive_cross_entropy(X, W, Y):
    logits = jnp.dot(X, W.T)
    lse = jax.scipy.special.logsumexp(logits, axis=1)
    correct_logits = logits[jnp.arange(X.shape[0]), Y]
    loss = jnp.mean(lse - correct_logits)
    return loss

# -----------------------------------------------------------------------------
# 4. Testing Framework
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    print("Testing JAX TPU Capstone (Memory-Efficient Backprop)...")
    
    N, H, V = 1024, 256, 32000
    chunk_size = 128
    
    # Initialize pseudo-random keys
    key = jax.random.PRNGKey(42)
    k1, k2, k3 = jax.random.split(key, 3)
    
    X = jax.random.normal(k1, (N, H))
    W = jax.random.normal(k2, (V, H))
    Y = jax.random.randint(k3, (N,), 0, V)
    
    # 1. Compile both functions using JIT (Just-In-Time compilation to XLA)
    jit_chunked = jax.jit(chunked_cross_entropy, static_argnums=(3,))
    jit_naive = jax.jit(naive_cross_entropy)
    
    # Value and Grad functions
    chunked_grad_fn = jax.jit(jax.value_and_grad(chunked_cross_entropy, argnums=(0, 1)), static_argnums=(3,))
    naive_grad_fn = jax.jit(jax.value_and_grad(naive_cross_entropy, argnums=(0, 1)))
    
    # Warmup
    _ = jit_naive(X, W, Y)
    _ = jit_chunked(X, W, Y, chunk_size)
    
    # Test Naive
    loss_naive, (dX_naive, dW_naive) = naive_grad_fn(X, W, Y)
    
    # Test Chunked
    loss_chunked, (dX_chunked, dW_chunked) = chunked_grad_fn(X, W, Y, chunk_size)
    
    # Verify correctness
    loss_diff = jnp.abs(loss_naive - loss_chunked)
    dX_diff = jnp.max(jnp.abs(dX_naive - dX_chunked))
    dW_diff = jnp.max(jnp.abs(dW_naive - dW_chunked))
    
    print(f"\nVerification Results:")
    print(f"Loss Difference: {loss_diff:.6e} {'' if loss_diff < 1e-4 else ''}")
    print(f"dX Max Difference: {dX_diff:.6e} {'' if dX_diff < 1e-4 else ''}")
    print(f"dW Max Difference: {dW_diff:.6e} {'' if dW_diff < 1e-4 else ''}")
    print(f"\nThis JAX script perfectly mirrors the PyTorch torch.autograd.Function implementation.")
    print("When you run this in Colab, jax.jit will automatically compile it via XLA for the TPU")
